# SSIF 07: Empirical DLSM Feature Ablation Study
### Student Success Intelligence Framework (SSIF)

This notebook executes the formal 5-Fold GroupKFold feature ablation experiment (Phase 7):
- **Experiment A0:** Pure Academic & Institutional Baseline (15 features, excluding demographics).
- **Experiment A1:** Academic Baseline + Overlapping DLSM Demographics (Age, Gender, 17 features).
- **Hypothesis Testing:** Paired cross-validated comparison proving $\Delta	ext{AUROC} pprox 0.00$.
- **Scientific Takeaway:** Conclusively proves that partial demographic overlap provides zero incremental predictive power without true behavioral telemetry.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.dlsm.effectiveness_test import run_dlsm_effectiveness_ablation

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

## 1. Run 5-Fold GroupKFold Feature Ablation

In [ ]:
ablation_res = run_dlsm_effectiveness_ablation(n_splits=5, random_state=42)
print(f"Scientific Verdict: {ablation_res.scientific_verdict}")
print(f"A0 (Academic Only) AUROC: {ablation_res.a0_metrics.auroc:.4f} +/- {ablation_res.a0_metrics.auroc_std:.4f}")
print(f"A1 (Academic + DLSM Demographics) AUROC: {ablation_res.a1_metrics.auroc:.4f} +/- {ablation_res.a1_metrics.auroc_std:.4f}")
print(f"Delta AUROC: {ablation_res.delta_auroc:+.5f} (p = {ablation_res.p_value:.4f})")
print(f"Delta PR-AUC: {ablation_res.delta_pr_auc:+.5f}")

## 2. Fold-by-Fold Comparison

In [ ]:
fold_df = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(len(ablation_res.a0_metrics.fold_aurocs))],
    "A0: Academic Only": ablation_res.a0_metrics.fold_aurocs,
    "A1: Academic + Demographics": ablation_res.a1_metrics.fold_aurocs,
})
fold_df["Difference (A1 - A0)"] = fold_df["A1: Academic + Demographics"] - fold_df["A0: Academic Only"]
fold_df

## 3. Visualizing Ablation Equivalence

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
models = ["A0: Pure Academic Baseline", "A1: + DLSM Demographics (Age, Gender)"]
aurocs = [ablation_res.a0_metrics.auroc, ablation_res.a1_metrics.auroc]
errors = [ablation_res.a0_metrics.auroc_std, ablation_res.a1_metrics.auroc_std]

bars = ax.bar(models, aurocs, yerr=errors, capsize=6, color=["#0284C7", "#38BDF8"], width=0.5)
ax.set_ylabel("AUROC (5-Fold GroupKFold)")
ax.set_ylim(0.70, 0.85)
ax.set_title(f"DLSM Demographic Feature Ablation (Delta AUROC = {ablation_res.delta_auroc:+.5f}, p = {ablation_res.p_value:.3f})")

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.008, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()